## Unificación de datasets y imputación valores faltantes

In [130]:
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns

In [131]:
df_orders = pd.read_parquet('../Dataset limpiado/ordenes_de_trabajo.parquet')
df_age_equipment = pd.read_parquet('../Dataset limpiado/age_equipment_limp.parquet')
df_mechanic = pd.read_parquet('../Dataset limpiado/mechanic_limp.parquet')

In [132]:
df_orders.head(3)

,equipment,operation,wo_type,base_model,problem_desc,failure_desc,cause_desc,action_desc,mechanic,date,downtime_gross,pit_coverage,real_downtime,parts_cost,repair_hours,waiting_hours,changed_pcs,days_between_failures
0,100268,BIND LEG,Corrective Maintenance,VF2500,It sews with defect / Cose con defecto,Machine head failure stopping sewing / Falla ...,Misadjusted machine head teeth / Dientes de ca...,Adjust machine head teeth / Ajustar dientes de...,Jaime Rivera,2023-10-03,1.25,0.0,1.25,16.81,1.25,0.0,1,81.0
1,100268,BIND LEG,Corrective Maintenance,VF2500,It sews with defect / Cose con defecto,Machine head failure stopping sewing / Falla ...,Misadjusted machine head teeth / Dientes de ca...,Adjust machine head teeth / Ajustar dientes de...,Jaime Rivera,2023-10-03,1.25,0.0,1.25,16.81,1.25,0.0,1,0.0
2,100268,BIND LEG,Corrective Maintenance,VF2500,It sews with defect / Cose con defecto,Machine head failure stopping sewing / Falla ...,Misadjusted machine head needle bar / Barra de...,Adjust machine head teeth / Ajustar dientes de...,Jaime Rivera,2024-01-17,3.00,3.0,0.00,12.50,2.00,1.0,2,106.0


In [133]:
df_age_equipment.head(3)

,equipment,age
0,1014059,22
1,1156767,19
2,1011680,22


In [134]:
df_mechanic.head(3)

,mechanic,antiquity
0,RIVAS TORRES CARLOS ANTONIO,19
1,Marvin Betancourth,18
2,JUAN BAAK CHIN,17


### Unimos 3 datasets a uno.

In [135]:
df = pd.merge(df_orders, df_age_equipment, on='equipment', how='left')
df = pd.merge(df, df_mechanic, on='mechanic', how='left')

In [136]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 141573 entries, 0 to 141572
Data columns (total 20 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   equipment              141573 non-null  int64         
 1   operation              141573 non-null  category      
 2   wo_type                141573 non-null  category      
 3   base_model             141573 non-null  category      
 4   problem_desc           141573 non-null  category      
 5   failure_desc           141573 non-null  category      
 6   cause_desc             141573 non-null  category      
 7   action_desc            141573 non-null  category      
 8   mechanic               141573 non-null  object        
 9   date                   141573 non-null  datetime64[ns]
 10  downtime_gross         141573 non-null  float64       
 11  pit_coverage           141573 non-null  float64       
 12  real_downtime          141573 non-null  floa

In [137]:
df['mechanic'] = df['mechanic'].astype('category')

In [138]:
variables = pd.DataFrame(columns=['Variable', 'Número de valores unicos', 'Tipo','Valores'])
for i, var in enumerate(df.columns):
    variables.loc[i] = [var, df[var].nunique(), df[var].dtype,df[var].unique().tolist()]

variables

,Variable,Número de valores unicos,Tipo,Valores
0,equipment,6974,int64,"[100268, 100312, 100452, 100884, 101632, 10163..."
1,operation,59,category,"[BIND LEG, BARTACK, SET SLEEVE, TOP STITCH, SE..."
2,wo_type,7,category,"[Corrective Maintenance, Breakdown, Preventati..."
3,base_model,119,category,"[VF2500, LT2-B872, AZ8403, W664, EX5214, VG270..."
4,problem_desc,24,category,"[It sews with defect / Cose con defecto, It do..."
5,failure_desc,38,category,[Machine head failure stopping sewing / Falla...
6,cause_desc,280,category,[Misadjusted machine head teeth / Dientes de c...
7,action_desc,278,category,[Adjust machine head teeth / Ajustar dientes d...
8,mechanic,185,category,"[Jaime Rivera, JERONIMO FUNEZ, Fabit Aranda, M..."
9,date,501,datetime64[ns],"[2023-10-03 00:00:00, 2024-01-17 00:00:00, 202..."


### Revisión de valores nulos despues de haber unido datasets.

In [139]:
# Valores nulos por variable
nulos = df.isnull().sum()

# Calcular el porcentaje de nulos
nulos_porcentaje = (nulos / len(df)) * 100

null_info = pd.DataFrame({
    'Variable': nulos.index,
    'Nulos': nulos.values,
    'Porcentaje de Nulos': nulos_porcentaje.values
})

null_info

,Variable,Nulos,Porcentaje de Nulos
0,equipment,0,0.000000
1,operation,0,0.000000
2,wo_type,0,0.000000
3,base_model,0,0.000000
4,problem_desc,0,0.000000
5,failure_desc,0,0.000000
6,cause_desc,0,0.000000
7,action_desc,0,0.000000
8,mechanic,0,0.000000
9,date,0,0.000000


#### Observamos que el porecentaje de valores faltantes de la variable **age** de maquina es de 0.68 por ciento, asi que<br>la eliminación de datos faltantes no comprometerá la validez estadística.

In [140]:
df = df.dropna(subset=['age'])
df.shape

(140602, 20)

In [141]:
# Valores nulos por variable
nulos = df.isnull().sum()

# Calcular el porcentaje de nulos
nulos_porcentaje = (nulos / len(df)) * 100

null_info = pd.DataFrame({
    'Variable': nulos.index,
    'Nulos': nulos.values,
    'Porcentaje de Nulos': nulos_porcentaje.values
})

null_info

,Variable,Nulos,Porcentaje de Nulos
0,equipment,0,0.000000
1,operation,0,0.000000
2,wo_type,0,0.000000
3,base_model,0,0.000000
4,problem_desc,0,0.000000
5,failure_desc,0,0.000000
6,cause_desc,0,0.000000
7,action_desc,0,0.000000
8,mechanic,0,0.000000
9,date,0,0.000000


#### Se ve que variable **antiquity** de mecánico tiene nulos. 

#### Aplicamos el KNN Imputer de libreria sklearn para rellenar los NaN basándose en la antigüedad de los 5 mecánicos que tengan estadísticas más similares

Agrumapos dataset por nombres de mecánicos

In [142]:
df_mecanicos = df.groupby('mechanic').agg(
    antiquity = ('antiquity', 'max'), 
    num_fallas = ('mechanic', 'count'),         
    downtime_gross = ('downtime_gross', 'sum'),
    real_downtime = ('real_downtime', 'sum'),
    repair_hours = ('repair_hours', 'sum'), 
    waiting_hours = ('waiting_hours', 'sum'), 
    changed_pcs = ('changed_pcs', 'sum'),
    parts_cost = ('parts_cost', 'sum'),                           
    days_between_failures = ('days_between_failures', 'mean'),   
    pit_coverage = ('pit_coverage', 'sum')
)

In [143]:
df_mecanicos.head(5)

,antiquity,num_fallas,downtime_gross,real_downtime,repair_hours,waiting_hours,changed_pcs,parts_cost,days_between_failures,pit_coverage
mechanic,,,,,,,,,,
Pedro Miguel Martinez Linares,NaN,942,2428.350000,1451.600000,1381.950000,1046.400000,1503,17124.77,2.606157,976.75
ALLAN RIVERA,18.0,58,25.400000,25.400000,24.383333,1.016667,18,578.49,15.637931,0.00
ANGEL MATAMOROS,18.0,146,62.733333,62.733333,60.066667,2.666667,36,1494.83,27.821918,0.00
Abel Alberto Ramirez Rivera,10.0,1262,2640.766667,608.056667,1341.533333,1299.233333,1359,7487.90,11.692552,2032.71
Abidail Alexander Garcia,1.0,102,187.333333,35.493333,172.750000,14.583333,180,1629.47,3.000000,151.84


Vemos que faltan los valores antiquity de 9 mecánicos.

In [144]:
# Valores nulos por variable
nulos = df_mecanicos.isnull().sum()

# Calcular el porcentaje de nulos
nulos_porcentaje = (nulos / len(df_mecanicos)) * 100

null_info = pd.DataFrame({
    'Variable': nulos.index,
    'Nulos': nulos.values,
    'Porcentaje de Nulos': nulos_porcentaje.values
})

null_info

,Variable,Nulos,Porcentaje de Nulos
0,antiquity,9,4.864865
1,num_fallas,0,0.000000
2,downtime_gross,0,0.000000
3,real_downtime,0,0.000000
4,repair_hours,0,0.000000
5,waiting_hours,0,0.000000
6,changed_pcs,0,0.000000
7,parts_cost,0,0.000000
8,days_between_failures,0,0.000000
9,pit_coverage,0,0.000000


Nombres de mecánicos.

In [145]:
mecanicos_nulos = df_mecanicos[df_mecanicos['antiquity'].isnull()]['antiquity'].index
mecanicos_nulos

CategoricalIndex([' Pedro Miguel Martinez Linares', 'Hugo Hernández',
                  'José Maria Canul Tec', 'José Valencia',
                  'Juan Manuel Interiano Zaldaña',
                  'Noé Alexander Caballero Flores', 'Solmar Elías Mezquita',
                  'Walter Bladimir Garcia Patiño', 'Walter Patiño'],
                 categories=[' Pedro Miguel Martinez Linares', 'ALLAN RIVERA', 'ANGEL MATAMOROS', 'Abel Alberto Ramirez Rivera', ..., 'William Alexander Arevalo Leiva', 'Willian Tejada', 'Wilmer Zamora', 'Xavier Jose Mendoza'], ordered=False, dtype='category', name='mechanic')

Aplicamos KNN imputer al dataset de mecánicos.

In [146]:
# Guardamos el índice (nombres de mecánicos) y las columnas
indices = df_mecanicos.index
columnas = df_mecanicos.columns

# Escalamos: Pasamos todo al rango [0, 1]
scaler = MinMaxScaler()
df_scaled = scaler.fit_transform(df_mecanicos)

# Imputación: Llenamos los NaN
imputer = KNNImputer(n_neighbors=5)
df_imputed_scaled = imputer.fit_transform(df_scaled)

# D. Desescalado: Volvemos a los valores originales (pesos, horas, etc.)
df_final_array = scaler.inverse_transform(df_imputed_scaled)

# E. Reconstrucción del DataFrame
df_mecanicos_limpio = pd.DataFrame(df_final_array, columns=columnas, index=indices)

Mecánicos con valores nans de antiquity rellenados por KNN imputer

In [147]:
df_mecanicos_limpio[df_mecanicos_limpio.index.isin(mecanicos_nulos)]

,antiquity,num_fallas,downtime_gross,real_downtime,repair_hours,waiting_hours,changed_pcs,parts_cost,days_between_failures,pit_coverage
mechanic,,,,,,,,,,
Pedro Miguel Martinez Linares,4.4,942.0,2428.350000,1451.600000,1381.950000,1046.400000,1503.0,17124.77,2.606157,976.75
Hugo Hernández,5.4,1112.0,1101.683333,36.733333,819.266667,282.416667,803.0,14592.98,21.833633,1064.95
José Maria Canul Tec,8.0,706.0,1034.900000,795.000000,851.600000,183.300000,861.0,15376.96,27.497167,239.90
José Valencia,8.4,640.0,579.666667,136.396667,498.116667,81.550000,530.0,7811.57,28.564062,443.27
Juan Manuel Interiano Zaldaña,6.8,911.0,1711.783333,175.103333,872.250000,839.533333,862.0,3928.77,11.268935,1536.68
Noé Alexander Caballero Flores,14.2,684.0,808.833333,284.063333,584.750000,224.083333,621.0,7340.68,20.881579,524.77
Solmar Elías Mezquita,17.2,97.0,155.250000,26.530000,90.783333,64.466667,88.0,1201.16,43.690722,128.72
Walter Bladimir Garcia Patiño,15.0,200.0,370.100000,81.910000,209.083333,161.016667,210.0,2270.53,41.870000,288.19
Walter Patiño,7.6,283.0,181.916667,38.916667,161.566667,20.350000,153.0,3305.37,13.279152,143.00


### Creamos el dataset de fallas limpio y sin valores faltantes

In [ ]:
#Eliminamos la columna antiquity con valores faltanres
df.drop(columns='antiquity', inplace=True)

# Creamos al dataset de fallos con valores nans imputados
df = pd.merge(df, df_mecanicos_limpio['antiquity'], left_on='mechanic', right_index=True, how='left')

#### Agregamos calendario: dia, semana, mes

In [149]:
if 'dia' not in df.columns:
    df['dia'] = df['date'].dt.to_period('D')

if 'semana' not in df.columns:
    df['semana'] = df['date'].dt.to_period('W')

if 'mes' not in df.columns:
    df['mes'] = df['date'].dt.to_period('M')

#### Guardamos el dataset unificado y sin valores faltantes.

In [150]:
df.to_parquet('../Dataset limpiado/dataset unificado.parquet')